In [0]:
%run "/Workspace/Users/jogesh.rajiyan@axahealth.co.uk/Utilities"

In [0]:
%sql
DROP view if exists transcripts;
create or replace temporary table Transcripts as
select
  `id` correlation_id,
  get_json_object(`Metadata`, '$.telephonyv1.attributes.external_call_id') conversationid,
  get_json_object(
    `Metadata`,
    '$.telephonyv1.attributes.analytics_session_start_time'
  ) sessionstarttime,
  chattimestamp,
  get_json_object(`Metadata`, '$.telephonyv1.agent_upn') Agentemail,
  transcript
from
  axahealth_dataplatform_pr_silver.call_summarisation.summaries_health_sensitive;

SELECT * FROM Transcripts where conversationid = '9cf6ecf5-410d-4866-b308-99e0840c9350';

In [0]:
%sql
create or replace temporary view transcriptswithparty as
select
  a.*,
  b.PartyKey,
  b.RowEffectiveDate,
  case
    when b.RowExpiryDate is null then current_date()
    else b.RowExpiryDate
  end as RowExpiryDate2
from
  transcripts a
    left join axahealth_dataplatform_pr_gold.base_layer.party_contact_mechanism b
      on lower(a.agentemail) = lower(b.ContactMechanismIdentifier);

SELECT * FROM transcriptswithparty;

In [0]:
%sql
Create or replace temporary view transcriptswithparty3 as
select
  *
from
  transcriptswithparty
where
  sessionstarttime BETWEEN RowEffectiveDate AND RowExpiryDate2

# Call Data 

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE v_genesys_allinboundcalls as
SELECT
  a.ConversationId,
  date_format(a.ConversationEndTimestamp, 'yyyy-MM-dd') as CallDate,
  a.ConversationStartTimestamp,
  a.ConversationEndTimestamp,
  a.OriginatingDirection,
  b.TelephonyConversationSessionKey as SessionKey,
  b.SessionId,
  b.ParentSessionId,
  regexp_replace(b.DialedNumberIdentificationService, 'tel:\\+44', '0') as DNIS,
  regexp_replace(b.CallerNumber, 'tel:\\+44', '0') as CallerNumber,
  b.SessionDirection,
  b.SessionParticipantRole as ParticipantRole,
  b.SessionType,
  b.AgentId,
  agt.AgentName,
  agt.EmailAddress,
  c.QueueId,
  c.SegmentType,
  c.DisconnectReason,
  c.SegmentStartTimestamp,
  c.SegmentEndTimestamp
FROM
  axahealth_dataplatform_pr_gold.base_layer.telephony_conversation a
    INNER JOIN axahealth_dataplatform_pr_gold.base_layer.telephony_conversation_session b
      ON a.TelephonyConversationKey = b.TelephonyConversationKey
    INNER JOIN axahealth_dataplatform_pr_gold.base_layer.dim_telephony_agent agt
      ON b.TelephonyAgentKey = agt.TelephonyAgentKey
    INNER JOIN axahealth_dataplatform_pr_gold.base_layer.telephony_conversation_session_segments c
      ON b.TelephonyConversationSessionKey = c.TelephonyConversationSessionKey
WHERE
  a.OriginatingDirection = 'inbound';

SELECT
  *
FROM
  v_genesys_allinboundcalls;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_genesys_conv_callernumber as
select distinct
  ConversationId,
  CallerNumber
from
  v_genesys_allinboundcalls
where
  ParticipantRole = 'customer'
  and sessiontype = 'voice'
  and sessiondirection = 'inbound';

SELECT
  *
FROM
  v_genesys_conv_callernumber;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE V_genesys_queuelevel_metrics as
SELECT
  TelephonyConversationSessionKey as SessionKey,
  SUM(
    CASE
      WHEN MetricName = 'nOffered' THEN MetricValue
      ELSE NULL
    END
  ) as nOffered,
  SUM(
    CASE
      WHEN MetricName = 'nConnected' THEN MetricValue
      ELSE NULL
    END
  ) as nConnected,
  SUM(
    CASE
      WHEN MetricName = 'tAcd' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tAcd,
  SUM(
    CASE
      WHEN MetricName = 'tAbandon' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tAbandon,
  SUM(
    CASE
      WHEN MetricName = 'tAnswered' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tAnswered,
  SUM(
    CASE
      WHEN MetricName = 'tIvr' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tIvr,
  SUM(
    CASE
      WHEN MetricName = 'nOverSla' THEN MetricValue
      ELSE NULL
    END
  ) as nOverSla,
  SUM(
    CASE
      WHEN MetricName = 'tShortAbandon' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tShortAbandon
FROM
  axahealth_dataplatform_pr_gold.base_layer.telephony_conversation_session_metrics
WHERE
  MetricName in (
    'nOffered', 'nConnected', 'tAcd', 'tAbandon', 'tAnswered', 'tIvr', 'nOverSla', 'tShortAbandon'
  )
GROUP BY
  TelephonyConversationSessionKey;

SELECT
  *
FROM
  V_genesys_queuelevel_metrics;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE V_genesys_agentlevel_metrics as
SELECT
  TelephonyConversationSessionKey as SessionKey,
  SUM(
    CASE
      WHEN MetricName = 'nTransferred' THEN MetricValue
      ELSE NULL
    END
  ) as nTransferred,
  SUM(
    CASE
      WHEN MetricName = 'nBlindTransferred' THEN MetricValue
      ELSE NULL
    END
  ) as nBlindTransferred,
  SUM(
    CASE
      WHEN MetricName = 'nConsultTransferred' THEN MetricValue
      ELSE NULL
    END
  ) as nConsultTransferred,
  SUM(
    CASE
      WHEN MetricName = 'nConsult' THEN MetricValue
      ELSE NULL
    END
  ) as nConsult,
  SUM(
    CASE
      WHEN MetricName = 'nOutbound' THEN MetricValue
      ELSE NULL
    END
  ) as nOutbound,
  SUM(
    CASE
      WHEN MetricName = 'tAnswered' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tAnswered,
  SUM(
    CASE
      WHEN MetricName = 'tAlert' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tAlert,
  SUM(
    CASE
      WHEN MetricName = 'tContacting' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tContacting,
  SUM(
    CASE
      WHEN MetricName = 'tDialing' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tDialing,
  SUM(
    CASE
      WHEN MetricName = 'tTalk' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tTalk,
  SUM(
    CASE
      WHEN MetricName = 'tHeld' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tHeld,
  SUM(
    CASE
      WHEN MetricName = 'tAcw' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tAcw,
  SUM(
    CASE
      WHEN MetricName = 'tHandle' THEN MetricValue / 1000
      ELSE NULL
    END
  ) as tHandle
FROM
  axahealth_dataplatform_pr_gold.base_layer.telephony_conversation_session_metrics
WHERE
  MetricName in (
    'nTransferred',
    'nBlindTransferred',
    'nConsultTransferred',
    'nConsult',
    'nOutbound',
    'tAnswered',
    'tAlert',
    'tContacting',
    'tDialing',
    'tTalk',
    'tHeld',
    'tAcw',
    'tHandle'
  )
GROUP BY
  TelephonyConversationSessionKey;

SELECT
  *
FROM
  V_genesys_agentlevel_metrics;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE V_genesys_queueoffered AS
SELECT
  ConversationId,
  CallDate,
  ConversationStartTimestamp,
  ConversationEndTimestamp,
  OriginatingDirection,
  AgentId,
  AgentName,
  AgentEmail,
  QueueId,
  SessionDirection,
  SUM(CallOffered) as CallOffered,
  SUM(CallofferedinSL) as CallofferedinSL,
  SUM(CallAbandon) as CallAbandon,
  SUM(CallAnswered) as CallAnswered,
  SUM(CallAnsweredInSL) as CallAnsweredInSL,
  SUM(CallQueueWaitTimeSecs) as CallQueueWaitTimeSecs,
  SUM(CallAnsweredTimeSecs) as CallAnsweredTimeSecs
FROM
  (
    SELECT DISTINCT
      ConversationId,
      CallDate,
      ConversationStartTimestamp,
      ConversationEndTimestamp,
      OriginatingDirection,
      QueueId,
      AgentId,
      AgentName,
      AgentEmail,
      SessionKey,
      SessionDirection,
      CallOffered,
      (CallOffered - CallShortAbandoned) CallofferedinSL,
      CallAbandon,
      CallAnswered,
      (CallAnswered - CallOverSla) CallAnsweredInSL,
      CallQueueWaitTimeSecs,
      CallAnsweredTimeSecs
    FROM
      (
        SELECT DISTINCT
          A.ConversationId,
          A.CallDate,
          A.ConversationStartTimestamp,
          A.ConversationEndTimestamp,
          A.OriginatingDirection,
          A.QueueId,
          A.AgentId,
          A.AgentName,
          A.EmailAddress AS AgentEmail,
          A.SessionKey,
          A.SessionDirection,
          CASE
            WHEN B.nOffered IS NOT NULL THEN 1
            ELSE 0
          END as CallOffered,
          CASE
            WHEN B.tAbandon IS NOT NULL THEN 1
            ELSE 0
          END as CallAbandon,
          CASE
            WHEN B.tAnswered IS NOT NULL THEN 1
            ELSE 0
          END as CallAnswered,
          CASE
            WHEN B.tAcd IS NOT NULL THEN B.tAcd
            ELSE 0
          END as CallQueueWaitTimeSecs,
          CASE
            WHEN B.tAnswered IS NOT NULL THEN B.tAnswered
            ELSE 0
          END as CallAnsweredTimeSecs,
          CASE
            WHEN B.nOverSla IS NOT NULL THEN 1
            ELSE 0
          END as CallOverSla,
          CASE
            WHEN B.tShortAbandon IS NOT NULL THEN 1
            ELSE 0
          END as CallShortAbandoned
        FROM
          v_genesys_allinboundcalls A
            LEFT JOIN V_genesys_queuelevel_metrics as B
              ON A.SessionKey = B.SessionKey
            LEFT JOIN V_genesys_agentlevel_metrics as C
              ON A.SessionKey = C.SessionKey
        WHERE
          A.QueueId IS NOT NULL
          AND A.SessionDirection = 'inbound'
          AND A.SessionType = 'voice'
      )
  )
GROUP BY
  ConversationId,
  CallDate,
  ConversationStartTimestamp,
  ConversationEndTimestamp,
  OriginatingDirection,
  AgentId,
  agentname,
  agentemail,
  QueueId,
  SessionDirection;

SELECT
  *
FROM
  V_genesys_queueoffered;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE V_genesys_agent_talk_hold_wrap AS
SELECT
  ConversationId,
  CallDate,
  ConversationStartTimestamp,
  ConversationEndTimestamp,
  OriginatingDirection,
  SessionDirection,
  AgentId,
  AgentName,
  AgentEmail,
  QueueId,
  SUM(CallTransfer) as CallTransfer,
  SUM(BlindTransfer) as BlindTransfer,
  SUM(ConsultTransfer) as ConsultTransfer,
  SUM(TotalConsults) as TotalConsults,
  SUM(Outbound) as Outbound,
  SUM(Answered) as Answered,
  SUM(Handled) as Handled,
  SUM(Talk) as Talk,
  SUM(Hold) as Hold,
  SUM(Wrap) as Wrap,
  SUM(Alert) as Alert,
  SUM(Contacting) as Contacting,
  SUM(Dialing) as Dialing,
  SUM(AlertTimeSecs) as AlertTimeSecs,
  SUM(ContactingTimeSecs) as ContactingTimeSecs,
  SUM(DialingTimeSecs) as DialingTimeSecs,
  SUM(AnsweredTimeSecs) as AnsweredTimeSecs,
  SUM(TalkTimeSecs) as TalkTimeSecs,
  SUM(HoldTimeSecs) as HoldTimeSecs,
  SUM(WrapTimeSecs) as WrapTimeSecs,
  SUM(HandleTimeSecs) as HandleTimeSecs
FROM
  (
    SELECT DISTINCT
      A.ConversationId,
      A.CallDate,
      A.ConversationStartTimestamp,
      A.ConversationEndTimestamp,
      A.OriginatingDirection,
      A.AgentId as AgentId,
      A.AgentName as AgentName,
      A.EmailAddress as AgentEmail,
      A.QueueId as QueueId,
      A.SessionKey,
      A.SessionDirection,
      CASE
        WHEN B.nTransferred IS NOT NULL THEN 1
        ELSE 0
      END as CallTransfer,
      CASE
        WHEN B.nBlindTransferred IS NOT NULL THEN 1
        ELSE 0
      END as BlindTransfer,
      CASE
        WHEN B.nConsultTransferred IS NOT NULL THEN 1
        ELSE 0
      END as ConsultTransfer,
      CASE
        WHEN B.nConsult IS NOT NULL THEN 1
        ELSE 0
      END as TotalConsults,
      CASE
        WHEN B.nOutbound IS NOT NULL THEN 1
        ELSE 0
      END as Outbound,
      CASE
        WHEN B.tAnswered > 0 THEN 1
        ELSE 0
      END as Answered,
      CASE
        WHEN B.tHandle > 0 THEN 1
        ELSE 0
      END as Handled,
      CASE
        WHEN B.tTalk IS NOT NULL THEN 1
        ELSE 0
      END as Talk,
      CASE
        WHEN B.tHeld IS NOT NULL THEN 1
        ELSE 0
      END as Hold,
      CASE
        WHEN B.tAcw IS NOT NULL THEN 1
        ELSE 0
      END as Wrap,
      CASE
        WHEN B.tAlert IS NOT NULL THEN 1
        ELSE 0
      END as Alert,
      CASE
        WHEN B.tContacting IS NOT NULL THEN 1
        ELSE 0
      END as Contacting,
      CASE
        WHEN B.tDialing IS NOT NULL THEN 1
        ELSE 0
      END as Dialing,
      CASE
        WHEN B.tAlert IS NOT NULL THEN B.tContacting
        ELSE 0
      END as AlertTimeSecs,
      CASE
        WHEN B.tContacting IS NOT NULL THEN B.tContacting
        ELSE 0
      END as ContactingTimeSecs,
      CASE
        WHEN B.tDialing IS NOT NULL THEN B.tDialing
        ELSE 0
      END as DialingTimeSecs,
      CASE
        WHEN B.tAnswered IS NOT NULL THEN B.tAnswered
        ELSE 0
      END as AnsweredTimeSecs,
      CASE
        WHEN B.tTalk IS NOT NULL THEN B.tTalk
        ELSE 0
      END as TalkTimeSecs,
      CASE
        WHEN B.tHeld IS NOT NULL THEN B.tHeld
        ELSE 0
      END as HoldTimeSecs,
      CASE
        WHEN B.tAcw IS NOT NULL THEN B.tAcw
        ELSE 0
      END as WrapTimeSecs,
      CASE
        WHEN B.tHandle IS NOT NULL THEN B.tHandle
        ELSE 0
      END as HandleTimeSecs
    FROM
      v_genesys_allinboundcalls A
        INNER JOIN V_genesys_agentlevel_metrics as B
          ON A.SessionKey = B.SessionKey
    WHERE
      A.AgentId IS NOT NULL
      AND A.DisconnectReason IS NOT NULL
      AND A.SessionType = 'voice'
  )
GROUP BY
  ConversationId,
  CallDate,
  ConversationStartTimestamp,
  ConversationEndTimestamp,
  OriginatingDirection,
  SessionDirection,
  AgentId,
  agentname,
  agentemail,
  QueueId;

SELECT
  *
FROM
  V_genesys_agent_talk_hold_wrap;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE v_genesys_agentSessionTime as
SELECT
  ConversationId,
  SessionType,
  SessionDirection,
  AgentId,
  AgentName,
  EmailAddress as AgentEmail,
  QueueId,
  min(SegmentStartTimestamp) as SessionStartTime,
  max(SegmentEndTimestamp) as SessionEndTime
from
  v_genesys_allinboundcalls
where
  ParticipantRole = 'agent'
group by
  ConversationId,
  SessionType,
  SessionDirection,
  AgentId,
  AgentName,
  EmailAddress,
  QueueId;

SELECT
  *
FROM
  v_genesys_agentSessionTime;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE v_genesys_callbacks as
WITH v_genesys_customer_callbackrequest AS (
  SELECT
    *
  FROM
    v_genesys_allinboundcalls
  WHERE
    ParticipantRole in ('customer', 'user')
    and SessionType = 'callback'
),
v_genesys_queue_callbackrequest AS (
  SELECT
    *
  FROM
    v_genesys_allinboundcalls
  WHERE
    ParticipantRole = 'acd'
    and SessionType = 'callback'
),
v_genesys_agent_callback AS (
  SELECT
    *
  FROM
    v_genesys_allinboundcalls
  WHERE
    ParticipantRole = 'agent'
    and SessionType = 'callback'
)
SELECT
  ConversationId,
  CallDate,
  CallerNumber,
  ConversationStartTimestamp,
  ConversationEndTimestamp,
  OriginatingDirection,
  SessionDirection,
  QueueId,
  AgentId,
  AgentName,
  AgentEmail,
  SUM(CallbackRequested) as CallbackRequested,
  SUM(CallbackAnswered) as CallbackAnswered
FROM
  (
    SELECT DISTINCT
      A.ConversationId,
      A.CallDate,
      A.CallerNumber,
      A.ConversationStartTimestamp,
      A.ConversationEndTimestamp,
      A.OriginatingDirection,
      A.SessionKey as CallbackTriggeredSessionKey,
      B.SessionKey as CallbackOfferedSessionKey,
      C.SessionKey as CallbackAnsweredSessionKey,
      A.SessionDirection,
      coalesce(C.QueueId, coalesce(B.QueueId, A.QueueId)) as QueueId,
      coalesce(C.AgentId, coalesce(B.AgentId, A.AgentId)) as AgentId,
      coalesce(C.AgentName, coalesce(B.AgentName, A.AgentName)) as AgentName,
      coalesce(C.EmailAddress, coalesce(B.EmailAddress, A.EmailAddress)) as AgentEmail,
      CASE
        WHEN A.SessionKey IS NOT NULL THEN 1
        ELSE 0
      END as CallbackRequested,
      CASE
        WHEN E.tAnswered IS NOT NULL THEN 1
        ELSE 0
      END as CallbackAnswered
    FROM
      v_genesys_customer_callbackrequest A
        LEFT JOIN v_genesys_queue_callbackrequest B
          ON A.ConversationId = B.ConversationId
          and A.SessionId = B.ParentSessionId
        LEFT JOIN v_genesys_agent_callback C
          ON A.ConversationId = C.ConversationId
          and A.SessionId = C.ParentSessionId
        LEFT JOIN V_genesys_queuelevel_metrics as D
          ON A.SessionKey = D.SessionKey
        LEFT JOIN V_genesys_queuelevel_metrics as E
          ON C.SessionKey = E.SessionKey
  )
GROUP BY
  ConversationId,
  CallDate,
  CallerNumber,
  ConversationStartTimestamp,
  ConversationEndTimestamp,
  OriginatingDirection,
  SessionDirection,
  QueueId,
  AgentId,
  AgentName,
  AgentEmail;

SELECT
  *
FROM
  v_genesys_callbacks;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE A AS
SELECT
  a.ConversationId,
  a.CallDate,
  a.ConversationStartTimestamp,
  a.ConversationEndTimestamp,
  a.OriginatingDirection,
  d.CallerNumber,
  a.AgentId,
  a.AgentName,
  a.AgentEmail,
  A.QueueId,
  c.SessionStartTime as AgentSessionStartTime,
  c.SessionEndTime as AgentSessionEndTime,
  b.TalkTimeSecs,
  b.HandleTimeSecs
FROM
  V_genesys_queueoffered a
    LEFT JOIN V_genesys_agent_talk_hold_wrap b
      on a.ConversationId = b.ConversationId
      and a.AgentId = b.AgentId
      and a.QueueId = b.QueueId
      and a.SessionDirection = b.SessionDirection
    LEFT JOIN v_genesys_agentSessionTime c
      on b.ConversationId = c.ConversationId
      and b.AgentId = c.AgentId
      and b.QueueId = c.QueueId
      and c.SessionType = 'voice'
      and b.SessionDirection = c.SessionDirection
    LEFT JOIN v_genesys_conv_callernumber d
      on a.ConversationId = d.ConversationId
WHERE
  a.CallAnswered > 0;

SELECT
  *
FROM
  A;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE B AS
SELECT
  a.ConversationId,
  a.CallDate,
  a.ConversationStartTimestamp,
  a.ConversationEndTimestamp,
  a.OriginatingDirection,
  d.CallerNumber,
  a.AgentId,
  a.AgentName,
  a.AgentEmail,
  A.QueueId,
  c.SessionStartTime as AgentSessionStartTime,
  c.SessionEndTime as AgentSessionEndTime,
  b.TalkTimeSecs,
  b.HandleTimeSecs
FROM
  v_genesys_callbacks a
    LEFT JOIN V_genesys_agent_talk_hold_wrap b
      on a.ConversationId = b.ConversationId
      and a.AgentId = b.AgentId
      and a.QueueId = b.QueueId
      and a.SessionDirection = b.SessionDirection
    LEFT JOIN v_genesys_agentSessionTime c
      on b.ConversationId = c.ConversationId
      and b.AgentId = c.AgentId
      and c.SessionType = 'callback'
      and b.SessionDirection = c.SessionDirection
    LEFT JOIN v_genesys_conv_callernumber d
      on a.ConversationId = d.ConversationId
WHERE
  a.CallbackAnswered > 0;

SELECT
  *
FROM
  B;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_genesy_TestCallsExclusion AS
SELECT DISTINCT
  Key,
  CAST(TRIM(REPLACE(Reason, 'Migration Datetime:', '')) as TIMESTAMP) StartTime
FROM
  axahealth_dataplatform_pr_gold.base_layer.dim_exclusion
where
  SystemID = 29
  and KeyType = 'TelephonyQueueName';

SELECT
  *
FROM
  v_genesy_TestCallsExclusion;

In [0]:
%sql
CREATE or replace temporary TABLE genesys_agent_interactions_modified_v3 as
SELECT
  ConversationId,
  CallDate,
  CallerNumber,
  ConversationStartTimestamp,
  ConversationEndTimestamp,
  SessionType,
  a.QueueId,
  tq.QueueName,
  dimd.DivisionName,
  coalesce(tq.BusinessAreaLevel1, 'Others') as BusinessAreaLevel1,
  a.AgentId,
  a.AgentName,
  a.agentemail,
  AgentSessionStartTime,
  AgentSessionEndTime,
  IsCallbackExists,
  TalkTimeSecs,
  HandleTimeSecs
FROM
  (
    SELECT
      A.ConversationId,
      A.CallDate,
      A.ConversationStartTimestamp,
      A.ConversationEndTimestamp,
      'voice' as SessionType,
      A.OriginatingDirection,
      A.CallerNumber,
      A.AgentId,
      A.AgentName,
      A.AgentEmail,
      A.QueueId,
      A.AgentSessionStartTime,
      A.AgentSessionEndTime,
      CASE
        WHEN b.ConversationID IS NOT NULL THEN 'Yes'
        ELSE 'No'
      END as IsCallbackExists,
      A.TalkTimeSecs,
      A.HandleTimeSecs
    FROM
      A
        LEFT JOIN B
          ON a.ConversationId = b.ConversationID
          and a.QueueId = b.QueueId
    UNION ALL
    SELECT
      ConversationId,
      CallDate,
      ConversationStartTimestamp,
      ConversationEndTimestamp,
      'callback' as SessionType,
      OriginatingDirection,
      CallerNumber,
      AgentId,
      AgentName,
      AgentEmail,
      QueueId,
      AgentSessionStartTime,
      AgentSessionEndTime,
      'Yes' as IsCallbackExists,
      B.TalkTimeSecs,
      B.HandleTimeSecs
    FROM
      B
  ) a
    INNER JOIN axahealth_dataplatform_pr_gold.base_layer.dim_telephony_queue tq
      ON A.QueueId = tq.QueueId
      AND A.CallDate >= tq.VersionStartDate
      AND A.CallDate < coalesce(tq.VersionEndDate, to_date(current_date()))
    LEFT JOIN axahealth_dataplatform_pr_gold.base_layer.dim_telephony_agent ta
      ON A.AgentId = ta.AgentId
      AND A.CallDate >= ta.VersionStartDate
      AND A.CallDate < coalesce(ta.VersionEndDate, to_date(current_date()))
    INNER JOIN v_genesy_TestCallsExclusion E
      ON tq.QueueName LIKE CONCAT(E.Key, '%')
      AND A.ConversationStartTimestamp >= E.StartTime
    INNER JOIN axahealth_dataplatform_pr_gold.base_layer.dim_telephony_division dimd
      on dimd.TelephonyDivisionKey = tq.TelephonyDivisionKey
Order by
  conversationid;

SELECT
  *
FROM
  genesys_agent_interactions_modified_v3
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW genesys_verified_modified AS
(
  SELECT DISTINCT
    v.InteractionId,
    MembershipNumber
  FROM
    axahealth_dataplatform_pr_gold.base_layer.telephony_interaction_verification as v
  where
    v.telephonyconversationkey is not null
    and membershipnumber is not null
);

SELECT
  *
FROM
  genesys_verified_modified
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW genesys_contact_modified_v2 AS
(
  SELECT
    call.*,
    verified.membershipnumber,
    "GENESYS" as SystemName
  FROM
    genesys_agent_interactions_modified_v3 as call
      left join genesys_verified_modified as verified
        on call.ConversationId = verified.InteractionId
  WHERE
    call.BusinessAreaLevel1 IN ('UKSI', 'Large Corp', 'Fertility')
    --AND QueueName <> 'H_MCLM_AHP_PreAuth_QUE'
    AND CallDate >= '2025-07-01'
  --AND CallDate <= '2026-02-28'
  ORDER BY
    call.ConversationId
);

SELECT
  *
FROM
  genesys_contact_modified_v2
LIMIT 10;

In [0]:
%sql
select distinct
  QueueName,
  count(*)
from
  genesys_contact_modified_v2
where
  QueueName like '%AHP%'
group by
  QueueName

In [0]:
%sql
create or replace temporary table callswithparty as
select distinct
  a.*,
  date(a.ConversationStartTimestamp) as ConversationDate,
  b.PartyKey,
  b.RowEffectiveDate,
  case
    when b.RowExpiryDate is null then current_date()
    else b.RowExpiryDate
  end as RowExpiryDate2
from
  genesys_contact_modified_v2 a
    left join axahealth_dataplatform_pr_gold.base_layer.party_system_identifier b
      on a.agentid = b.SystemUserID;

SELECT
  *
FROM
  callswithparty
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE callswithparty3 AS
SELECT
  *
FROM
  callswithparty
where
  ConversationDate BETWEEN RowEffectiveDate AND RowExpiryDate2;

SELECT
  *
FROM
  callswithparty3
LIMIT 10;

In [0]:
%sql
WITH filtered_agents AS (
  SELECT
    a.agentid
  FROM
    callswithparty a
  WHERE
    a.PartyKey IS NOT NULL
    AND a.ConversationDate BETWEEN a.RowEffectiveDate AND a.RowExpiryDate2
  GROUP BY
    a.agentid
),
total_agents AS (
  SELECT
    COUNT(DISTINCT agentid) AS total_count
  FROM
    callswithparty
),
agents_with_party AS (
  SELECT
    COUNT(DISTINCT agentid) AS valid_agents_count
  FROM
    filtered_agents
)
SELECT
  (valid_agents_count * 100.0) / total_count AS percentage_with_party
FROM
  agents_with_party,
  total_agents;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE TelephonyConversationBase AS
SELECT a.ConversationId
,a.MembershipNumber
,c.ClaimNumber
,a.CallerNumber
,a.CallDate
,a.ConversationStartTimestamp
,a.ConversationEndTimestamp
, LAG(a.ConversationStartTimestamp) OVER(Partition by c.ClaimNumber ORDER BY a.ConversationStartTimestamp) AS PreviousCallTimestamp
, timestampdiff(second,
LAG(a.ConversationStartTimestamp) OVER(Partition by c.ClaimNumber ORDER BY a.ConversationStartTimestamp),a.ConversationStartTimestamp
) AS GapFromPreviousCallSecs
,a.AgentId
,a.AgentName
,a.AgentEmail
,a.QueueId
,a.QueueName
,a.BusinessAreaLevel1
,a.SessionType
,a.AgentSessionStartTime
,a.AgentSessionEndTime
,a.IsCallBackExists
,a.TalkTimeSecs
,a.HandleTimeSecs
,a.SystemName
,a.conversationdate
,a.partykey
,a.roweffectivedate
,a.rowexpirydate2
,b.correlation_id
,b.chattimestamp
,b.sessionstarttime
,b.transcript
,d.OpenDate as ClaimOpenDate
,d.CurrentCondition
,d.CurrentConditionCategory
,case
    when d.CurrentConditionCategory = 'MSK' then 1
    else 0
  end as MSKClaim
,d.IsManagedServices
,d.System
FROM callswithparty3 a
INNER JOIN transcriptswithparty3 b
ON a.conversationid = b.conversationid
AND a.partykey = b.partykey
inner join notesummaries2 c
    on b.correlation_id = c.note_correlation_id
inner join axahealth_dataplatform_pd_gold.base_layer.claim d
    on c.claimnumber = d.ClaimReference 
WHERE a.ConversationStartTimestamp IS NOT NULL AND a.ConversationId IS NOT NULL AND IsManagedServices = 'No';

SELECT * FROM telephonyconversationbase ORDER BY ClaimNumber,ConversationStartTimestamp
;

In [0]:
%sql
SELECT ConversationId, COUNT(DISTINCT agentid)
FROM telephonyconversationbase GROUP BY 1 HAVING COUNT(DISTINCT agentid) > 1


In [0]:
%sql
create or replace temporary table TelephonyMemberDetails as
WITH memdetails1 AS (
    select distinct
  a.ClaimNumber,
  g.dateofbirth,
  g.AgeCurrent,
  g.gender,
  g.relationship,
  g.JoinDate,
  g.Postcode,
  g.PremiumAnnualGrossIPT,
  g.PremiumAnnualNetIPT,
  g.HistoryID,
  g.CancellationDate
from
  TelephonyConversationBase a
    Left Join axahealth_dataplatform_pr_gold.base_layer.claim f
      on a.ClaimNumber = f.ClaimReference
    left join axahealth_dataplatform_pr_gold.base_layer.claim_invoice_link cil
      on f.ClaimKey = cil.ClaimKey
      and f.SystemID = cil.SystemID
      and f.memberid = cil.memberid
    left join axahealth_dataplatform_pr_gold.base_layer.members g
      on cil.MemberVersionId = g.MemberVersionId
      and cil.memberid = g.memberid

),
maxmemrow AS (
    SELECT claimnumber
    ,max(HistoryId) AS maxid
    FROM memdetails1
    group by claimnumber
),
memdetails2 AS (
    SELECT distinct b.*
    ,SUBSTRING_INDEX(postcode, ' ', 1) AS PostcodePrefix
    FROM maxmemrow a
    INNER JOIN memdetails1 b
    ON a.claimnumber = b.claimnumber
    AND a.maxid = b.historyid
)

select distinct
  a.ClaimNumber,
  b.dateofbirth,
  b.JoinDate,
  b.cancellationdate,
  b.Relationship,
  b.PremiumAnnualGrossIPT, -- member level from member table -- member paying
  b.PremiumAnnualNetIPT,
  b.AgeCurrent,
  b.Gender,
  b.Postcode,
  b.PostcodePrefix,
  c.UkRegion
from
  TelephonyConversationBase a
    Left Join memdetails2 b
      on a.claimnumber = b.claimnumber
    LEFT JOIN axahealth_dataplatform_pr_gold.base_layer.dim_geography c
    ON b.postcodeprefix = c.PostCode
;


select
  *
from
  MemberDetails ;

In [0]:
%sql
Create or replace temporary table TelephonyComplainttoContact as
select
  DISTINCT a.*,
  b.casekey,
  b.ComplaintArea
from
  TelephonyConversationBase a
    left join ComplainttoClaim2 b
      on a.ClaimNumber = b.claimNumber;

select
  *
from
  TelephonyComplainttoContact;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY table TelephonycomplaintRankedResponses2 AS
WITH complaintRankedResponses AS (
  SELECT
    a.*,
    b.ReceiptDate AS ReceiptDateTime,
    DATEDIFF(second, a.conversationstarttimestamp, b.ReceiptDate) AS timediffseconds,
    ROW_NUMBER() OVER (
        PARTITION BY a.conversationid
        ORDER BY
          ABS(DATEDIFF(second, a.conversationstarttimestamp, b.ReceiptDate)),
          b.ReceiptDate, -- Tie-breaker for same time difference
          b.CaseSK -- Additional tie-breaker for deterministic results
      ) AS rn,
    -- Also assign a row number per case key to enforce one case per conversation
    ROW_NUMBER() OVER (
        PARTITION BY b.CaseSK
        ORDER BY ABS(DATEDIFF(second, a.conversationstarttimestamp, b.ReceiptDate))
      ) AS case_rank
  FROM
    TelephonyComplainttoContact a
      LEFT JOIN axahealth_dataplatform_pr_silver.respond.dim_case b
        ON a.CaseKey = b.CaseSK
  WHERE
    b.ReceiptDate >= a.conversationstarttimestamp
)
SELECT
  *
FROM
  complaintRankedResponses
WHERE
  rn = 1
  AND case_rank = 1;

SELECT * FROM telephonycomplaintrankedresponses2;

In [0]:
%sql
     CREATE OR REPLACE TEMPORARY VIEW TelephonyExGratiaFollowingComplaint AS
     WITH ComplaintDates AS (
           SELECT
        a.claimnumber,
        c.conversationid,
        c.ReceiptDateTime,
        -- Find the first complaint date per claim
        MIN(c.ReceiptDateTime) OVER (PARTITION BY a.claimnumber) AS FirstComplaintDate,
        -- Find the next complaint date after the current complaint
        LEAD(c.ReceiptDateTime) OVER (PARTITION BY a.claimnumber ORDER BY c.ReceiptDateTime) AS NextComplaintDate
    FROM
        telephonyconversationbase a
    INNER JOIN
        telephonycomplaintrankedresponses2 c ON a.conversationid = c.conversationid
)
SELECT
    cd.claimnumber,
    cd.conversationid,
    -- Sum ex gratia payments between the first and next complaint dates
    SUM(
        CASE
            WHEN b.PaidDate >= cd.FirstComplaintDate
                 AND (b.PaidDate < cd.NextComplaintDate OR cd.NextComplaintDate IS NULL)
                 AND c.ReceiptDateTime IS NOT NULL
            THEN TRY_CAST(SPLIT_PART(b.TotalExGratiaPaid, '.', 1) AS BIGINT)
            ELSE 0
        END
    ) AS ExGratiaAmountPaid
FROM
    ComplaintDates cd
INNER JOIN
    ExGratiaClaimTotals b ON cd.claimnumber = b.ClaimNumber
LEFT JOIN
    telephonycomplaintrankedresponses2 c ON cd.conversationid = c.conversationid
GROUP BY
    cd.conversationid, cd.claimnumber;

SELECT * FROM TelephonyExGratiaFollowingComplaint;

In [0]:
%sql
Create or replace temporary view ClaimCost as 
select a.ClaimReference, sum(b.amountclaimed) as TotalClaimed, sum(b.AmountPaid)as ClaimTotalPaid
from axahealth_dataplatform_pr_gold.base_layer.claim a  
left join axahealth_dataplatform_pr_gold.base_layer.invoice b on a.ClaimID = b.ClaimID and a.SystemID = b.SystemID
group by a.ClaimReference;
select * from ClaimCost limit 10 


In [0]:
%sql
create or replace temporary view combined_telephony as
select distinct 
  a.*
 ,f.MemberId
  ,b.StandardisedSegment -- current segment
  ,c.PolicySubType -- current policy
  ,f.CurrentCondition
  ,f.CurrentConditionCategory
  ,f.System
  ,CASE WHEN f.CurrentConditionCategory = 'MSK' THEN 1 ELSE 0 END AS MSKClaim
  ,DATEDIFF(day,f.OpenDate, a.ConversationTimeStamp) DaysSinceClaimOpened
  ,FLOOR(a.GapFromPreviousMessageSecs/86400.00) AS DaysSincePreviousContact
  ,FLOOR(months_between(f.opendate, g.dateofbirth) / 12) AS ClaimantAge_ClaimOpenDate -- Age when Claim Opened
  , g.AgeCurrent
  , g.JoinDate
  --, DATEDIFF(day, g.JoinDate, GETDATE()) / 365.25 AS Tenure_Years
  ,CASE 
    WHEN g.CancellationDate IS NOT NULL 
        THEN DATEDIFF(day, g.JoinDate, g.CancellationDate) / 365.25
    ELSE DATEDIFF(day, g.JoinDate, GETDATE()) / 365.25
END AS Tenure_Years
  ,g.Relationship
  ,g.PremiumAnnualGrossIPT
  ,g.PremiumAnnualNetIPT
  ,g.Gender
  ,g.CancellationDate 
  ,g.UkRegion
  ,d.ClaimTotalPaid
  ,e.ExGratiaAmountPaid -- exgratia paid following complaint 
  ,h.ComplaintArea
  ,h.ReceiptDateTime as ComplaintReceiptDate
  ,v.VulnerableCustomer
  from
   mol_to_claim a
   Left Join Segmentation b on a.MembershipNumber = b.AXAMembershipNumber
    Left Join MaxPolicySubType c
    on a.ClaimNumber = c.ClaimReference
  Left Join ClaimCost d
    on a.ClaimNumber = d.ClaimReference
    Left Join ExGratiaFollowingComplaint e
      on a.claimnumber = e.claimnumber and a.casenumber = e.casenumber
    Left Join axahealth_dataplatform_pr_gold.base_layer.claim f
      on a.ClaimNumber = f.ClaimReference
    Left Join complaintrankedresponses2 h
    on a.casenumber = h.casenumber 
     left join MemberDetails g on a.ClaimNumber = g.ClaimNumber
     left join VulnerableCustomerClaims v on a.Claimnumber = v.Claimreference
   WHERE f.ClaimReference is not null --remove claims that are not in claims base layer table 
     Order by ClaimNumber, ConversationStartTimestamp asc ;

SELECT * FROM combined_mol WHERE casenumber = '11887956';
